# Full-System 06: YOLO + TrOCR + Qwen Hybrid Routing

This notebook writes a Kaggle-style CSV with columns `image,regions`.
It imports `fillpaper/inference.py` and keeps runtime settings aligned with that base file.


In [ ]:
# Install runtime packages required by this notebook if Kaggle image is missing them.
# If Internet is disabled, replace the pip specs below with local wheel paths.
import importlib.metadata
import importlib.util
import re
import subprocess
import sys

NEEDS_YOLO_PACKAGES = True
NEEDS_HPA_PACKAGES = True
NEEDS_QWEN_PACKAGES = True


def _version_tuple(version):
    nums = re.findall(r"\d+", str(version))
    return tuple(int(num) for num in nums[:3]) if nums else (0,)


def ensure_package(import_name, pip_spec, min_version=None, dist_name=None):
    needs_install = importlib.util.find_spec(import_name) is None
    if min_version is not None and not needs_install:
        try:
            current = importlib.metadata.version(dist_name or import_name)
            needs_install = _version_tuple(current) < _version_tuple(min_version)
        except importlib.metadata.PackageNotFoundError:
            needs_install = True
    if needs_install:
        print(f"Installing {pip_spec} ...", flush=True)
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U", pip_spec])


ensure_package("PIL", "pillow", dist_name="Pillow")

if NEEDS_YOLO_PACKAGES:
    ensure_package("cv2", "opencv-python-headless", dist_name="opencv-python-headless")
    ensure_package("ultralytics", "ultralytics")
    ensure_package("doclayout_yolo", "doclayout-yolo", dist_name="doclayout-yolo")

if NEEDS_QWEN_PACKAGES:
    ensure_package("transformers", "transformers>=4.57.0", min_version="4.57.0")
    ensure_package("accelerate", "accelerate")
    ensure_package("peft", "peft")
    ensure_package("qwen_vl_utils", "qwen-vl-utils", dist_name="qwen-vl-utils")
    ensure_package("bitsandbytes", "bitsandbytes>=0.46.1", min_version="0.46.1")
elif NEEDS_HPA_PACKAGES:
    ensure_package("transformers", "transformers")


In [ ]:
from pathlib import Path

# Edit these exact Kaggle paths before running.
RUN_NAME = "06_yolo_trocr_qwen_hybrid"
RUN_MODE = "hybrid"
PROMPT_VARIANT = "final"
USE_QWEN_LORA = True

INFERENCE_PY = Path("/kaggle/input/datasets/huylhn1810/inference-notebook/inference.py")
TEST_JSONL = Path("/kaggle/input/datasets/huylhn1810/htd-prefinetune-metadata/test.jsonl")
IMAGE_ROOT = Path("/kaggle/input/datasets/quii29/rukopys-dataset/train")

OUTPUT_DIR = Path("/kaggle/working") / RUN_NAME
OUTPUT_CSV = OUTPUT_DIR / f"{RUN_NAME}_submission.csv"

HPA_MODEL_DIR = Path("/kaggle/input/datasets/habao2603/trocr-finetune-for-paper/final_cyrillic_htr_model/final_cyrillic_htr_model")
QWEN_BASE_DIR = Path("/kaggle/input/models/qwen-lm/qwen-3-vl/transformers/8b-instruct/1")
QWEN_LORA_DIR = Path("/kaggle/input/datasets/huylhn1810/rukopys-qwen3vl-stage2b-finetune-dataset/qwen3vl_rukopys_stage2b_hardtype_aug_hybrid_prompt_v2/qwen3vl_rukopys_stage2b_hardtype_aug_lora_final")
YOLO_WEIGHTS = Path("/kaggle/input/datasets/huylhn1810/yolo-weight-final/Doclayout Yolo Final V4.1.pt")
YOLO_EXTRA_WEIGHTS = [Path("/kaggle/input/datasets/huylhn1810/yolo-weight-final/Doclayout Yolo Final V4.2.pt")]

NUM_GPUS = 2
GPU_DEVICES = ["cuda:0", "cuda:1"]

# Keep these aligned with fillpaper/inference.py unless intentionally testing a runtime parameter.
YOLO_BACKEND = "auto"
YOLO_IMG_SIZE = 1280
YOLO_CONF = 0.20
YOLO_MAX_DET = 300
YOLO_IOU_NMS = 0.55
YOLO_AGNOSTIC_NMS = True
YOLO_PAD_SCALE_X = 0.0
YOLO_PAD_SCALE_Y = 0.0
YOLO_DEDUP_IOU = 1.01
CROP_BATCH_SIZE = 2
MAX_PIXELS_CROP = 262_144
MAX_NEW_TOKENS_QWEN = 192
HPA_CROP_PAD_RATIO = 0.0
QWEN_CROP_PAD_RATIO = 0.04
LOG_EVERY = 10


In [ ]:

import contextlib
import csv
import gc
import importlib.util
import json
import multiprocessing as mp
import os
import sys
import traceback
import warnings
from pathlib import Path

TEXT_TYPES = {"handwritten", "printed", "formula", "table", "annotation"}
HPA_TYPES = {"handwritten", "printed", "annotation"}
QWEN_FORMULA_TABLE_TYPES = {"formula", "table"}
YOLO_MODES = {"detector_only", "hybrid", "trocr_all", "qwen_all", "hpa_only", "qwen_only"}
HPA_MODES = {"hybrid", "trocr_all", "hpa_only", "hpa_gt"}
QWEN_MODES = {"hybrid", "qwen_all", "qwen_only", "qwen_gt"}

SOURCE_HINTS = {
    "dictation": "Ukrainian dictation handwriting. Do not complete from canonical text; read only visible characters.",
    "archive": "Historical Ukrainian/Cyrillic document. Preserve old spelling; do not modernize.",
    "school": "School homework. It may contain corrections, teacher marks, formulas, and mixed handwriting/print.",
    "university": "University exam/coursework. It may contain formulas, tables, chemistry notation, and technical symbols.",
}
DEFAULT_SOURCE_HINT = "Read only visible characters from this crop."
SPECIAL_TEXT_MARKER_RULES = (
    "Use [illegible] only for unreadable words inside an otherwise legible text region. "
    "Use ~~word~~ for visible strikethrough and ~~old~~{new} for visible correction."
)
STAGE_B_GUARDRAILS = (
    "The final transcription must be supported by the crop. "
    "Do not complete missing words from source hint, language prior, or canonical dictation text. "
    "Do not translate, correct grammar, normalize spelling, expand abbreviations, summarize, "
    "or infer hidden/missing text. No JSON, no Markdown, no explanation."
)
GENERIC_QWEN_PROMPT = (
    "Transcribe the visible content exactly. Return only the transcription. "
    "Preserve visible text, mathematical symbols, table layout, punctuation, digits, corrections, "
    "line breaks, and original spelling when they are visible. Do not add content that is not visible."
)
TYPE_PROMPTS = {
    "handwritten": (
        "Transcribe the visible handwritten Ukrainian/Cyrillic text exactly. Preserve punctuation, "
        "line content, corrections, spelling mistakes, capitalization, digits, abbreviations, quotes, "
        "hyphens, visible spacing, and strikethrough markers."
    ),
    "printed": (
        "Transcribe the visible printed or typed Ukrainian/Cyrillic text exactly. Preserve punctuation, "
        "line content, corrections, capitalization, digits, abbreviations, quotes, hyphens, visible spacing, "
        "and strikethrough markers."
    ),
    "annotation": (
        "Read this short annotation, teacher mark, grade, correction, or numbering exactly. Return only "
        "the exact visible text."
    ),
    "formula": (
        "Read this standalone math, logic, vector, matrix, determinant, set/relation, statistics, physics, "
        "or chemistry expression exactly as written. Return only formula text, using LaTeX when it is the "
        "clearest representation and plain Unicode when it better matches the handwriting. Preserve visible "
        "symbols, indices, superscripts, subscripts, arrows, fractions, matrix/determinant structure, punctuation, "
        "numbering, and strikethrough/correction markers. Do not solve, simplify, normalize, explain, or convert "
        "old notation into a different style."
    ),
    "table": (
        "Read this table region exactly. Return only pipe-separated table text. Use one output line per visual row "
        "and | between cells. Preserve empty cells with empty fields, for example A||C. Preserve row order, "
        "column order, multi-word cell text, wrapped cell text, numbers, units, punctuation, dashes, and visible "
        "spelling mistakes. Do not infer missing cells, do not rebalance columns, do not summarize, and do not explain."
    ),
}


def read_jsonl(path):
    rows = []
    with Path(path).open('r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows


def write_submission_rows(rows, output_csv):
    output_csv = Path(output_csv)
    output_csv.parent.mkdir(parents=True, exist_ok=True)
    tmp = output_csv.with_suffix(output_csv.suffix + '.tmp')
    with tmp.open('w', encoding='utf-8', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=['image', 'regions'])
        writer.writeheader()
        for row in rows:
            writer.writerow(row)
    tmp.replace(output_csv)


def output_image_name(record):
    return Path(str(record.get('file_name') or record.get('image') or '')).name


def image_path_for_record(record):
    return IMAGE_ROOT / str(record['file_name'])


def load_inference_module():
    spec = importlib.util.spec_from_file_location(f'fillpaper_inference_base_{os.getpid()}', str(INFERENCE_PY))
    module = importlib.util.module_from_spec(spec)
    sys.modules[spec.name] = module
    spec.loader.exec_module(module)
    return module


def source_hint(source):
    return SOURCE_HINTS.get(str(source or '').strip().lower(), DEFAULT_SOURCE_HINT)


def prompt_for_variant(rtype, source, default_prompt):
    rtype = str(rtype or 'handwritten').strip().lower()
    type_prompt = TYPE_PROMPTS.get(rtype, default_prompt)
    if PROMPT_VARIANT == 'generic':
        return GENERIC_QWEN_PROMPT
    if PROMPT_VARIANT == 'type_specific':
        return '\n'.join([type_prompt, SPECIAL_TEXT_MARKER_RULES])
    if PROMPT_VARIANT == 'source_type':
        return '\n'.join([source_hint(source), type_prompt, SPECIAL_TEXT_MARKER_RULES])
    if PROMPT_VARIANT == 'source_guardrails':
        return '\n'.join([source_hint(source), default_prompt, SPECIAL_TEXT_MARKER_RULES, STAGE_B_GUARDRAILS])
    return '\n'.join([source_hint(source), type_prompt, SPECIAL_TEXT_MARKER_RULES, STAGE_B_GUARDRAILS])


def qwen_load_without_lora(self):
    if self.model is not None:
        return
    import torch
    from qwen_vl_utils import process_vision_info
    from transformers import AutoModelForImageTextToText, AutoProcessor, BitsAndBytesConfig

    if not self.cfg.device.startswith('cuda'):
        raise RuntimeError('Qwen3-VL branch requires CUDA for these Kaggle notebooks.')

    self.torch = torch
    self.process_vision_info = process_vision_info
    print(f'Loading base Qwen3-VL without LoRA: {self.cfg.qwen_base_dir}', flush=True)
    quantization_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type='nf4',
    )
    common_kwargs = dict(
        device_map={'': self.cfg.device},
        quantization_config=quantization_config,
        trust_remote_code=True,
        attn_implementation='sdpa',
        low_cpu_mem_usage=True,
    )
    try:
        self.model = AutoModelForImageTextToText.from_pretrained(
            str(self.cfg.qwen_base_dir), dtype=torch.float16, **common_kwargs
        )
    except TypeError:
        self.model = AutoModelForImageTextToText.from_pretrained(
            str(self.cfg.qwen_base_dir), torch_dtype=torch.float16, **common_kwargs
        )
    self.model.eval()
    self.processor = AutoProcessor.from_pretrained(str(self.cfg.qwen_base_dir), trust_remote_code=True)
    self._configure_processor()


def configure_inference_module(inf):
    inf.CROP_PROMPTS.update(TYPE_PROMPTS)
    default_prompt = inf.CROP_PROMPTS['default']

    def build_qwen_prompt_for_run(rtype, source=None):
        return prompt_for_variant(rtype, source, default_prompt)

    inf.build_qwen_prompt = build_qwen_prompt_for_run

    if not USE_QWEN_LORA:
        inf.QwenOcr.load = qwen_load_without_lora

    if RUN_MODE == 'trocr_all':
        inf.BAOHA_TYPES = set(TEXT_TYPES)
        inf.QWEN_TYPES = set()
    elif RUN_MODE == 'qwen_all':
        inf.BAOHA_TYPES = set()
        inf.QWEN_TYPES = set(TEXT_TYPES)
    elif RUN_MODE in {'qwen_only', 'qwen_gt'}:
        inf.BAOHA_TYPES = set()
        inf.QWEN_TYPES = set(QWEN_FORMULA_TABLE_TYPES)
    elif RUN_MODE in {'hpa_only', 'hpa_gt'}:
        inf.BAOHA_TYPES = set(HPA_TYPES)
        inf.QWEN_TYPES = set()
    elif RUN_MODE == 'hybrid':
        inf.BAOHA_TYPES = set(HPA_TYPES)
        inf.QWEN_TYPES = set(QWEN_FORMULA_TABLE_TYPES)


def build_runtime_config(inf, device):
    return inf.RuntimeConfig(
        input_dir=IMAGE_ROOT,
        output_dir=OUTPUT_DIR,
        hpa_model_dir=HPA_MODEL_DIR,
        qwen_base_dir=QWEN_BASE_DIR,
        qwen_lora_dir=QWEN_LORA_DIR,
        yolo_weights=YOLO_WEIGHTS,
        yolo_extra_weights=list(YOLO_EXTRA_WEIGHTS),
        device=device,
        yolo_backend=YOLO_BACKEND,
        yolo_img_size=YOLO_IMG_SIZE,
        yolo_conf=YOLO_CONF,
        yolo_max_det=YOLO_MAX_DET,
        yolo_iou_nms=YOLO_IOU_NMS,
        yolo_agnostic_nms=YOLO_AGNOSTIC_NMS,
        yolo_pad_scale_x=YOLO_PAD_SCALE_X,
        yolo_pad_scale_y=YOLO_PAD_SCALE_Y,
        yolo_dedup_iou=YOLO_DEDUP_IOU,
        crop_batch_size=CROP_BATCH_SIZE,
        max_pixels_crop=MAX_PIXELS_CROP,
        max_new_tokens_qwen=MAX_NEW_TOKENS_QWEN,
        hpa_crop_pad_ratio=HPA_CROP_PAD_RATIO,
        qwen_crop_pad_ratio=QWEN_CROP_PAD_RATIO,
        lazy_load=True,
    )


def gt_regions_from_record(inf, record):
    width = int(record['image_width'])
    height = int(record['image_height'])
    regions = []
    for region in record.get('regions', []):
        box = inf.clamp_xyxy(region.get('bbox'), width, height)
        if box is None:
            continue
        regions.append({'bbox': box, 'type': inf.normalize_type(region.get('type')), 'text': ''})
    return inf.sort_regions([inf.strip_region(r) for r in regions])


def detect_with_pil_source(inf, detector, image_path):
    detector.load()
    result_list = []
    from PIL import Image

    try:
        img_src = Image.open(image_path).convert('RGB')
    except Exception:
        img_src = str(image_path)

    for model in detector.models:
        results = model.predict(
            source=img_src,
            imgsz=detector.cfg.yolo_img_size,
            conf=detector.cfg.yolo_conf,
            max_det=detector.cfg.yolo_max_det,
            verbose=False,
            device=inf.yolo_predict_device(detector.cfg.device),
        )
        if results:
            result_list.append(results[0])
    if not result_list:
        return []

    has_boxes = any(getattr(result, 'boxes', None) is not None and len(result.boxes) > 0 for result in result_list)
    if not has_boxes:
        return []

    result = result_list[0]
    if hasattr(result, 'orig_shape') and result.orig_shape:
        img_h, img_w = result.orig_shape
    elif hasattr(img_src, 'size'):
        img_w, img_h = img_src.size
    else:
        with Image.open(image_path) as img:
            img_w, img_h = img.size
    return detector._postprocess(result_list, int(img_w), int(img_h))


def init_components(inf, cfg):
    detector = inf.YoloDetector(cfg) if RUN_MODE in YOLO_MODES else None
    hpa = inf.HpaOcr(cfg) if RUN_MODE in HPA_MODES else None
    qwen = inf.QwenOcr(cfg) if RUN_MODE in QWEN_MODES else None
    if detector is not None:
        detector.load()
    return detector, hpa, qwen


def infer_one_record(inf, detector, hpa, qwen, record):
    image_path = image_path_for_record(record)
    source = record.get('source')

    if RUN_MODE == 'empty_prediction':
        return []

    if RUN_MODE in {'hpa_gt', 'qwen_gt'}:
        regions = gt_regions_from_record(inf, record)
    else:
        regions = detect_with_pil_source(inf, detector, image_path)

    if not regions:
        return []
    if RUN_MODE == 'detector_only':
        return inf.sort_regions([inf.strip_region(r) for r in regions])

    if hpa is not None:
        hpa.ocr_regions(image_path, regions)
    if qwen is not None:
        qwen.ocr_regions(image_path, regions, source=source)

    return inf.sort_regions([inf.strip_region(region) for region in regions])


def worker_main(shard_index, device, shard_records, partial_csv):
    os.environ.setdefault('TRANSFORMERS_VERBOSITY', 'error')
    os.environ.setdefault('TRANSFORMERS_NO_ADVISORY_WARNINGS', '1')
    os.environ.setdefault('PYTORCH_ALLOC_CONF', 'expandable_segments:True')
    os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
    warnings.filterwarnings('ignore', message=r'.*Kwargs passed to `processor\.__call__` have to be in `processor_kwargs` dict.*')

    if str(device).startswith('cuda'):
        import torch
        cuda_index = int(str(device).split(':', 1)[1]) if ':' in str(device) else 0
        torch.cuda.set_device(cuda_index)
        torch.cuda.empty_cache()
        print(f'[shard {shard_index}] pinned cuda device={torch.cuda.current_device()}', flush=True)

    if RUN_MODE in QWEN_MODES and shard_index > 0:
        import time
        time.sleep(8 * shard_index)

    inf = load_inference_module()
    inf.configure_runtime()
    configure_inference_module(inf)
    cfg = build_runtime_config(inf, device)
    detector, hpa, qwen = init_components(inf, cfg)

    rows = []
    total = len(shard_records)
    print(f'[shard {shard_index}] start run={RUN_NAME} mode={RUN_MODE} device={device} records={total}', flush=True)
    for local_index, record in enumerate(shard_records, 1):
        image_name = output_image_name(record)
        try:
            regions = infer_one_record(inf, detector, hpa, qwen, record)
        except Exception as exc:
            print(f'[shard {shard_index}] failed {image_name}: {exc}', flush=True)
            traceback.print_exc()
            regions = []
        rows.append({'image': image_name, 'regions': json.dumps(regions, ensure_ascii=False)})
        if local_index == 1 or local_index % LOG_EVERY == 0 or local_index == total:
            print(f'[shard {shard_index}] {local_index}/{total} {image_name} regions={len(regions)}', flush=True)

    write_submission_rows(rows, partial_csv)
    print(f'[shard {shard_index}] wrote {partial_csv}', flush=True)
    gc.collect()


def merge_partials(records, partial_paths, output_csv):
    by_image = {}
    for path in partial_paths:
        with Path(path).open('r', encoding='utf-8', newline='') as f:
            for row in csv.DictReader(f):
                by_image[row['image']] = row['regions']
    rows = []
    for record in records:
        image = output_image_name(record)
        rows.append({'image': image, 'regions': by_image[image]})
    write_submission_rows(rows, output_csv)
    print(f'Wrote merged submission: {output_csv} rows={len(rows)}', flush=True)


def run_parallel():
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    records = read_jsonl(TEST_JSONL)
    print(f'RUN_NAME={RUN_NAME}')
    print(f'RUN_MODE={RUN_MODE}')
    print(f'PROMPT_VARIANT={PROMPT_VARIANT}')
    print(f'USE_QWEN_LORA={USE_QWEN_LORA}')
    print(f'records={len(records)}')
    print(f'output={OUTPUT_CSV}')

    shards = [records[i::NUM_GPUS] for i in range(NUM_GPUS)]
    partial_paths = [OUTPUT_DIR / f'{RUN_NAME}_shard{i}_partial.csv' for i in range(NUM_GPUS)]

    ctx = mp.get_context('fork')
    processes = []
    for shard_index, shard_records in enumerate(shards):
        process = ctx.Process(
            target=worker_main,
            args=(shard_index, GPU_DEVICES[shard_index], shard_records, partial_paths[shard_index]),
        )
        process.start()
        processes.append(process)

    failed = []
    for process in processes:
        process.join()
        if process.exitcode != 0:
            failed.append(process.exitcode)
    if failed:
        raise RuntimeError(f'Worker process failed with exit codes: {failed}')

    merge_partials(records, partial_paths, OUTPUT_CSV)


run_parallel()
